<H1>Create aggregated variables from a formatted dataset<H1>

<H2> Import packages <H2>

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import norm

<H2> Read and check the dataset as 'data' <H2> 

In [2]:
data = pd.read_csv('/Path/to/Combined_trial_data.csv')

In [3]:
data.head()

,study,experiment,id,session,trial_num,block,trial_num_within_block,set_size,change,response,rt,fixation_size,stim_size,stim_duration,retention_interval,distance_from_monitor,test_location_x_from_center,test_location_y_from_center
0,1,1,1,1,1,1,1,6,1,0,4344.281496,0.415,1.775,250,1000,60,-156.919732,95.561873
1,1,1,1,1,2,1,2,6,0,1,5743.139812,0.415,1.775,250,1000,60,259.327759,-115.628762
2,1,1,1,1,3,1,3,6,0,0,1431.351300,0.415,1.775,250,1000,60,-157.719064,178.484950
3,1,1,1,1,4,1,4,8,1,1,2025.997530,0.415,1.775,250,1000,60,91.568562,-98.117057
4,1,1,1,1,5,1,5,4,0,0,957.352518,0.415,1.775,250,1000,60,-196.886288,176.424749


In [4]:
#Uncomment if there is an extra column at the beginning
#data = data.drop(data.columns[[0]], axis = 1)
#data.head()

In [5]:
data['response'].isna().sum() #The response variable doesn't have any missing values

np.int64(0)

In [6]:
data['rt'].isna().sum() #The reaction time also doesn't have any missing values

np.int64(0)

<H2> Set function parameters <H2>

In [7]:
grouping_variables = ['id']
grouping_variables_string = "id" #Specify the variables used for grouping the dataset
group_by_set_size = False
filter_set_sizes = True #If True, input which set sizes to omit below
set_sizes_to_omit = [6]
filter_rows = True #If True, input the max rows per participant within each set size. Row numbers larger than this will be omitted.
max_row_number = 60
omit_rt_outliers = False #If true, extreme rt values will be omitted from the dataset before computing the reaction time summaries

<H2> Omit outlying reaction time values<H2>
    (if specified above) 

In [8]:
def remove_group_outliers(df, group_cols, value_col, iqr_multiplier=1.5):
    
    """
    Removes outliers from a dataframe based on group-specific IQR calculations.
    
    Parameters:
    - df: Pandas DataFrame
    - group_cols: List of column names to group by
    - value_col: Name of the numerical column to check for outliers
    - iqr_multiplier: Multiplier for IQR to define bounds (default 1.5)
    
    Returns:
    - Filtered dataframe with outliers omitted
    
    """
    
    # Calculate Q1, Q3, and IQR for each group
    q1 = df.groupby(group_cols)[value_col].transform('quantile', 0.25)
    q3 = df.groupby(group_cols)[value_col].transform('quantile', 0.75)
    iqr = q3 - q1
    
    # Define bounds
    lower_bound = q1 - (iqr_multiplier * iqr)
    upper_bound = q3 + (iqr_multiplier * iqr)
    
    # Set values that fall outside the bounds to na
    df[value_col] = np.where(df[value_col] < lower_bound, np.nan, df[value_col])
    df[value_col] = np.where(df[value_col] > upper_bound, np.nan, df[value_col])
    
    df = df.reset_index()
    
    return df

In [9]:
#Run the function on the data
#Call the function on the reaction time column, grouped by id
if omit_rt_outliers == True :
    data = remove_group_outliers(data, 'id', 'rt', iqr_multiplier=1.5)
else : 
    print("All reaction time values were included.")

All reaction time values were included.


In [10]:
data.shape

(827646, 18)

<H2>Trim the dataset<H2> (if/as specified above) 

The final dataset will contain the desired set sizes and number of trials per participant and set size.

In [11]:
def trim_dataset(df, filter_set_sizes, filter_rows):

    """
    Removes rows from a dataframe above the max row number specified.
    Removes specified set sizes from the dataframe
    
    Parameters:
    - df: Pandas DataFrame
    - trim_set_sizes: True or False. If False, the dataframe remains unaltered. 
    - trim_rows: True or False. If False, the dataframe remains unaltered. 
    
    Returns:
    - Filtered dataframe with specified rows and set sizes omitted. 
    """

    global filtered_data

    if filter_rows == True : 
        n = max_row_number # Keep the first n rows within each id and set size
        df = df.groupby(['id', 'set_size']).head(n)
        print('Row numbers >', max_row_number, 'within each set size were omitted.')

    else :
        print("All rows were included.")

    if filter_set_sizes == True : 
        for set_size in set_sizes_to_omit :
            df = df[df['set_size'] != set_size]
            print('Set sizes', set_sizes_to_omit, 'were omitted.')
    else  :
        print("All set sizes were included.")

    filtered_data = df.reset_index()   
    
    return filtered_data

In [12]:
trim_dataset(data, filter_set_sizes, filter_rows)

Row numbers > 60 within each set size were omitted.
Set sizes [6] were omitted.


,index,study,experiment,id,session,trial_num,block,trial_num_within_block,set_size,change,response,rt,fixation_size,stim_size,stim_duration,retention_interval,distance_from_monitor,test_location_x_from_center,test_location_y_from_center
0,3,1,1,1,1,4,1,4,8,1,1,2025.997530,0.415,1.775,250,1000,60,91.568562,-98.117057
1,4,1,1,1,1,5,1,5,4,0,0,957.352518,0.415,1.775,250,1000,60,-196.886288,176.424749
2,6,1,1,1,1,7,1,7,4,0,1,1164.270226,0.415,1.775,250,1000,60,245.682274,134.190635
3,7,1,1,1,1,8,1,8,4,1,0,1434.097588,0.415,1.775,250,1000,60,-256.036789,131.100334
4,8,1,1,1,1,9,1,9,8,1,1,1612.714707,0.415,1.775,250,1000,60,-103.364548,-222.759197
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
486235,827641,2,1,4052,1,116,1,116,4,0,0,700.000000,1.300,0.400,150,900,60,100.000000,300.000000
486236,827642,2,1,4052,1,117,1,117,4,1,1,444.000000,1.300,0.400,150,900,60,100.000000,300.000000
486237,827643,2,1,4052,1,118,1,118,4,0,0,638.000000,1.300,0.400,150,900,60,-200.000000,-200.000000
486238,827644,2,1,4052,1,119,1,119,4,1,1,1137.000000,1.300,0.400,150,900,60,200.000000,300.000000


In [13]:
filtered_data.shape #Confirm that the dimensions are as expected

(486240, 19)

In [14]:
filtered_data.head()

,index,study,experiment,id,session,trial_num,block,trial_num_within_block,set_size,change,response,rt,fixation_size,stim_size,stim_duration,retention_interval,distance_from_monitor,test_location_x_from_center,test_location_y_from_center
0,3,1,1,1,1,4,1,4,8,1,1,2025.997530,0.415,1.775,250,1000,60,91.568562,-98.117057
1,4,1,1,1,1,5,1,5,4,0,0,957.352518,0.415,1.775,250,1000,60,-196.886288,176.424749
2,6,1,1,1,1,7,1,7,4,0,1,1164.270226,0.415,1.775,250,1000,60,245.682274,134.190635
3,7,1,1,1,1,8,1,8,4,1,0,1434.097588,0.415,1.775,250,1000,60,-256.036789,131.100334
4,8,1,1,1,1,9,1,9,8,1,1,1612.714707,0.415,1.775,250,1000,60,-103.364548,-222.759197


In [15]:
data.head()

,study,experiment,id,session,trial_num,block,trial_num_within_block,set_size,change,response,rt,fixation_size,stim_size,stim_duration,retention_interval,distance_from_monitor,test_location_x_from_center,test_location_y_from_center
0,1,1,1,1,1,1,1,6,1,0,4344.281496,0.415,1.775,250,1000,60,-156.919732,95.561873
1,1,1,1,1,2,1,2,6,0,1,5743.139812,0.415,1.775,250,1000,60,259.327759,-115.628762
2,1,1,1,1,3,1,3,6,0,0,1431.351300,0.415,1.775,250,1000,60,-157.719064,178.484950
3,1,1,1,1,4,1,4,8,1,1,2025.997530,0.415,1.775,250,1000,60,91.568562,-98.117057
4,1,1,1,1,5,1,5,4,0,0,957.352518,0.415,1.775,250,1000,60,-196.886288,176.424749


<H2>Numeric variable (reaction time) </H2>

Compute the trial counts, mean, standard deviation, and quantile values of a numeric variable (rt).

In [16]:
def numeric_variable_stats(df, group_cols, value_col):
    
    """
    Parameters:
    - df: Pandas DataFrame
    - group_cols: the levels of the dataframe within which the variables will be computed. 
    - value_col: Expected to be reaction time.  
    
    Returns:
    - a dataframe summarizing the mean, standard deviation, and quantile values
    (0, .25, .50, .75, 1) within each level of a numeric variable
    within the original grouped DataFrame. 
    
    """
    
    global numeric_output
    
    df = df.groupby(group_cols)[value_col].describe(percentiles=[0.25, 0.5, 0.75]).reset_index()
    
    #Alternative method to get just the mean and SD
    #df.groupby(group_cols)[value_col].agg(mean="mean", std="std").reset_index()    

    if group_by_set_size == True :
        
        colnames = ['id', 'set_size', 'trial_count', 'rt_mean', 'rt_sd', 'rt_min', 'rt_q1',
                'rt_median', 'rt_q3', 'rt_max']
    else :
        colnames = ['id', 'trial_count', 'rt_mean', 'rt_sd', 'rt_min', 'rt_q1',
                'rt_median', 'rt_q3', 'rt_max']
        
    df = pd.DataFrame(df)
    df = df.set_axis(colnames, axis=1)

    numeric_output = df
    
    return numeric_output    

In [17]:
#Call the function on the dataset
numeric_variable_stats(filtered_data, grouping_variables, "rt")

,id,trial_count,rt_mean,rt_sd,rt_min,rt_q1,rt_median,rt_q3,rt_max
0,1,120.0,1182.821835,587.670328,522.394129,850.073460,1028.449882,1257.220832,4753.319327
1,2,120.0,912.584378,410.279908,346.817413,660.971880,824.501812,979.788072,3066.020889
2,3,120.0,878.829760,519.535754,439.670791,574.350781,693.634287,953.729406,3030.393356
3,4,120.0,1150.633437,625.986459,501.069991,753.336470,918.889555,1369.477404,4527.712615
4,5,120.0,935.776111,489.918361,334.818047,592.003827,765.913577,1120.274291,2630.756338
...,...,...,...,...,...,...,...,...,...
4047,4048,120.0,697.216667,235.614633,368.000000,537.500000,649.000000,776.250000,1895.000000
4048,4049,120.0,676.125000,249.324080,292.000000,527.750000,618.000000,760.250000,1716.000000
4049,4050,120.0,933.350000,394.959856,461.000000,687.750000,802.000000,1032.500000,2430.000000
4050,4051,120.0,1345.400000,744.267204,611.000000,884.750000,1127.500000,1597.500000,5398.000000


<H2> Categorical variables <H2>

Summarize the trial counts of each condition (combining change and response).

In [18]:
def categorical_variable_counts(df, group_cols, value_col):

    """
    Counts up the total number of hits, false alarms, correction rejections, and misses
    within each level of a grouped dataframe
    0 = same color
    1 = different color
    hits are trials where change = 1 and response = 1 
    false alarms are trials where change = 0 and response = 1 
    correct rejections are trials where change = 0 and response = 0
    misses are trials where change = 1 and response = 0
    
    Parameters:
    - df: Pandas DataFrame
    - group_cols: the levels of the dataframe within which the variables will be computed. 
    - value_col: Any will do, since the function only counts up the number of rows in each condition. 

    Returns:
    - a dataframe providing the total counts of hits, false alarms, correct rejections,
    and misses within the original grouped dataframe.
    
    """
    
    global counts

    permanent_cols = ['change', 'response']
    final_group_cols =  list([*group_cols, *permanent_cols])

    df = df.groupby(final_group_cols)[value_col].size().unstack(fill_value=0).unstack(fill_value=0)
    df = pd.DataFrame(df)
    df.columns = ['correct_rejections', 'misses', 'false_alarms', 'hits']

    counts = df.reset_index()

    return counts

In [19]:
#Call the function on the dataset
categorical_variable_counts(filtered_data, grouping_variables, "trial_num")

,id,correct_rejections,misses,false_alarms,hits
0,1,31,20,29,40
1,2,29,8,31,52
2,3,36,5,24,55
3,4,25,7,35,53
4,5,39,11,21,49
...,...,...,...,...,...
4047,4048,51,2,13,54
4048,4049,40,5,22,53
4049,4050,16,56,44,4
4050,4051,43,2,17,58


<H2> Create final analysis variables </H2>

In [20]:
def create_analysis_variables(df):

    """
    Creates the analysis variables
    accuracy is the # correct responses / # responses
    d' is the sensitivity to a true change, computed as z(hit rate) - z(false alarm rate)
    A' is a non-parametric measure of sensitivity. 
    K is a measure of capacity that approximates how many items are simultaneously 
    held in working memory. K can only be computed if the dataframe was grouped by set size. 
    Otherwise, an empty column is returned. 
    
    Parameters:
    - df: Pandas DataFrame expected to be the counts dataframe from the previous function
    
    Returns:
    - a dataframe providing the final variables for analysis, including
    the accuracy, d', A', response bias and K.  
    
    """
    
    global analysis_variables
    
    #Compute rate variables with a correction for extreme values
    df['accuracy'] = (df['hits'] + df['correct_rejections'] + .5)/(df['hits'] + df['correct_rejections'] + df['false_alarms'] + df['misses'] + 1)
    df['hit_rate'] = (df['hits'] + .5) /(df['hits'] + df['misses'] + 1)
    df['false_alarm_rate'] = (df['false_alarms'] + .5) / (df['correct_rejections'] + df['false_alarms'] + 1)
    df['correct_rejection_rate'] = (df['correct_rejections'] + .5) / (df['correct_rejections'] + df['false_alarms'] + 1)
    df['miss_rate'] = (df['misses'] + .5) / (df['hits'] + df['misses'] + 1)


    #Compute analysis variables

    #Estimation of d' (sensitivity to a true change)
    df['dprime'] = norm.ppf(df['hit_rate']) - norm.ppf(df['false_alarm_rate'])

    #A' is a non-parametric alternative to d'.
    df['aprime'] = .5 + (((df['hit_rate'] - df['false_alarm_rate']) * (1 + df['hit_rate'] - df['false_alarm_rate'])) / (4 * df['hit_rate'] * (1 -  df['false_alarm_rate'])))
    #Some people use this formula instead that reverses hits and false alarms if false alarms > hits: 
    #df['aprime'] = np.where(df['hit_rate'] >= df['false_alarm_rate'],
                            #(.5 + (((df['hit_rate'] - df['false_alarm_rate']) * (1 + df['hit_rate'] - df['false_alarm_rate'])) / (4 * df['hit_rate'] * (1 -  df['false_alarm_rate'])))),
                            #(.5 + (((df['false_alarm_rate'] - df['hit_rate']) * (1 + df['false_alarm_rate'] - df['hit_rate'])) / (4 * df['false_alarm_rate'] * (1 -  df['hit_rate'])))))
    
    #Response bias to choose 'different' with a correction for extreme values
    df['response_bias'] = (df['hits'] + df['false_alarms'] + .5)/(df['hits'] + df['correct_rejections'] + df['false_alarms'] + df['misses'] + 1)
    df['response_bias_probability'] = norm.ppf(df['hit_rate']) + norm.ppf(df['false_alarm_rate'])/2
    
    #Paschler's K, a measure of capacity that should remain stable across set sizes
    #This function has been modified such that values < 1 are diminishing positive fractions. 
    if group_by_set_size == True :
        df['k_temp'] = df['set_size'] * (df['hit_rate'] - df['false_alarm_rate']) / (1 - df['false_alarm_rate'])
        df['k'] = np.where(df['k_temp'] < 1, 
                           (1/(1+np.exp(-df['k_temp']))),
                           (df['k_temp']))
        df.drop(['k_temp'], axis = 1, inplace=True)
    else :
        df['k'] = ''

    analysis_variables = df

    return analysis_variables

In [21]:
#Run the function on the categorical dataset to create the new variables
create_analysis_variables(counts)

,id,correct_rejections,misses,false_alarms,hits,accuracy,hit_rate,false_alarm_rate,correct_rejection_rate,miss_rate,dprime,aprime,response_bias,response_bias_probability,k
0,1,31,20,29,40,0.590909,0.663934,0.483607,0.516393,0.336066,0.464329,0.655203,0.574380,0.402673,
1,2,29,8,31,52,0.673554,0.860656,0.516393,0.483607,0.139344,1.042166,0.777966,0.690083,1.103822,
2,3,36,5,24,55,0.756198,0.909836,0.401639,0.598361,0.090164,1.588852,0.851968,0.657025,1.215193,
3,4,25,7,35,53,0.648760,0.877049,0.581967,0.418033,0.122951,0.953433,0.760583,0.731405,1.263826,
4,5,39,11,21,49,0.731405,0.811475,0.352459,0.647541,0.188525,1.262036,0.818629,0.582645,0.694001,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4047,4048,51,2,13,54,0.871901,0.956140,0.207692,0.792308,0.043860,2.522008,0.931856,0.557851,1.300326,
4048,4049,40,5,22,53,0.772727,0.906780,0.357143,0.642857,0.093220,1.687288,0.865283,0.623967,1.138129,
4049,4050,16,56,44,4,0.169421,0.073770,0.729508,0.270492,0.926230,-2.059598,-2.328283,0.400826,-1.142609,
4050,4051,43,2,17,58,0.838843,0.959016,0.286885,0.713115,0.040984,2.301891,0.910846,0.623967,1.458131,


<H2> Combine and save the full dataset </H2>

In [22]:
#Bind the categorical variable results to the rt variables
final_analysis_variables = pd.concat([numeric_output, analysis_variables], axis=1).drop_duplicates()

In [23]:
if group_by_set_size == True :
    final_analysis_variables = final_analysis_variables[['id', 'set_size', 'rt_mean', 'rt_sd', 'trial_count', 'rt_min', 'rt_q1', 'rt_median', 'rt_q3', 'rt_max', 'hits', 'false_alarms', 
           'correct_rejections', 'misses', 'accuracy', 'hit_rate', 'false_alarm_rate', 'correct_rejection_rate', 'miss_rate',
           'dprime', 'aprime', 'response_bias', 'response_bias_probability', 'k']]
else : final_analysis_variables = final_analysis_variables[['id', 'rt_mean', 'rt_sd', 'trial_count', 'rt_min', 'rt_q1', 'rt_median', 'rt_q3', 'rt_max', 'hits', 'false_alarms', 
           'correct_rejections', 'misses', 'accuracy', 'hit_rate', 'false_alarm_rate', 'correct_rejection_rate', 'miss_rate',
           'dprime', 'aprime', 'response_bias', 'response_bias_probability', 'k']]
final_analysis_variables = final_analysis_variables.loc[:, ~final_analysis_variables.columns.duplicated()]

In [24]:
final_analysis_variables.head()

,id,rt_mean,rt_sd,trial_count,rt_min,rt_q1,rt_median,rt_q3,rt_max,hits,...,accuracy,hit_rate,false_alarm_rate,correct_rejection_rate,miss_rate,dprime,aprime,response_bias,response_bias_probability,k
0,1,1182.821835,587.670328,120.0,522.394129,850.073460,1028.449882,1257.220832,4753.319327,40,...,0.590909,0.663934,0.483607,0.516393,0.336066,0.464329,0.655203,0.574380,0.402673,
1,2,912.584378,410.279908,120.0,346.817413,660.971880,824.501812,979.788072,3066.020889,52,...,0.673554,0.860656,0.516393,0.483607,0.139344,1.042166,0.777966,0.690083,1.103822,
2,3,878.829760,519.535754,120.0,439.670791,574.350781,693.634287,953.729406,3030.393356,55,...,0.756198,0.909836,0.401639,0.598361,0.090164,1.588852,0.851968,0.657025,1.215193,
3,4,1150.633437,625.986459,120.0,501.069991,753.336470,918.889555,1369.477404,4527.712615,53,...,0.648760,0.877049,0.581967,0.418033,0.122951,0.953433,0.760583,0.731405,1.263826,
4,5,935.776111,489.918361,120.0,334.818047,592.003827,765.913577,1120.274291,2630.756338,49,...,0.731405,0.811475,0.352459,0.647541,0.188525,1.262036,0.818629,0.582645,0.694001,


In [25]:
#Write the dataset to a csv file with the grouping variables in the title. 
print('analysis variables were grouped by', grouping_variables)
print('filtering by set size was set to', filter_set_sizes, 'and trimming rows was set to', filter_rows)
print('set sizes', set_sizes_to_omit, 'were listed in set_sizes_to_omit and row number', max_row_number, 'was listed as the max_row_number within each set size.')
print('The dataset with trials has dimensions:', filtered_data.shape)
print('The final dataset with variables has dimensions:', final_analysis_variables.shape)
filename = 'analysis_variables_grouped_by_' + grouping_variables_string + '.csv'

#Write the csv file with the analysis variables
final_analysis_variables.to_csv(filename)

#Option to write the trimmed trial-level dataset to a csv file:
filtered_data.to_csv("filtered_trial_data.csv")

analysis variables were grouped by ['id']
filtering by set size was set to True and trimming rows was set to True
set sizes [6] were listed in set_sizes_to_omit and row number 60 was listed as the max_row_number within each set size.
The dataset with trials has dimensions: (486240, 19)
The final dataset with variables has dimensions: (4052, 23)
